## Assignment 1: Logistic Regression
Welcome to week one of this specialization. You will learn about logistic regression. Concretely, you will be implementing logistic regression for sentiment analysis on tweets. Given a tweet, you will decide if it has a positive sentiment or a negative one. Specifically you will: 

* Learn how to extract features for logistic regression given some text
* Implement logistic regression from scratch
* Apply logistic regression on a natural language processing task
* Test using your logistic regression
* Perform error analysis

### Import Functions and Data

In [21]:
import nltk
from os import getcwd
import unit_tests_w1 as ut
import numpy as np
import pandas as pd
from nltk.corpus import twitter_samples 
from utils import process_tweet, build_freqs
from sklearn.model_selection import train_test_split

nltk.download('twitter_samples')
nltk.download('stopwords')

[nltk_data] Downloading package twitter_samples to
[nltk_data]     C:\Users\mrinm\AppData\Roaming\nltk_data...
[nltk_data]   Package twitter_samples is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mrinm\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
filePath = f"{getcwd()}/../tmp2/"
nltk.data.path.append(filePath)


### Prepare the Data
* The `twitter_samples` contains subsets of five thousand positive_tweets, five thousand negative_tweets, and the full set of 10,000 tweets.  
    * If you used all three datasets, we would introduce duplicates of the positive tweets and negative tweets.  
    * You will select just the five thousand positive tweets and five thousand negative tweets.

In [3]:
# select the set of positive and negative tweets
all_positive_tweets = twitter_samples.strings('positive_tweets.json')
all_negative_tweets = twitter_samples.strings('negative_tweets.json')

* Train test split: 20% will be in the test set, and 80% in the training set.

In [4]:
all_positive_tweets_df = pd.DataFrame({'tweets': all_positive_tweets, 'label': [1]*len(all_positive_tweets)})
all_negative_tweets_df = pd.DataFrame({'tweets': all_negative_tweets, 'label': [0]*len(all_negative_tweets)})

df_tweets = pd.concat([all_positive_tweets_df, all_negative_tweets_df], ignore_index=True, axis = 0)

train_set, test_set = train_test_split(df_tweets, test_size = 0.2, random_state = 42)

In [5]:
train_x = train_set['tweets'].tolist()
test_x = test_set['tweets'].tolist()
train_y = train_set['label'].values.reshape(train_set['label'].values.size, 1)
test_y = test_set['label'].values.reshape(test_set['label'].values.size, 1)

In [6]:
# Print the shape train and test sets
print("train_y.shape = " + str(train_y.shape))
print("test_y.shape = " + str(test_y.shape))

train_y.shape = (8000, 1)
test_y.shape = (2000, 1)


In [7]:
# create frequency dictionary
freqs = build_freqs(train_x, train_y)

# check the output
print("type(freqs) = " + str(type(freqs)))
print("len(freqs) = " + str(len(freqs.keys())))

type(freqs) = <class 'dict'>
len(freqs) = 11449


### Process Tweet
The given function 'process_tweet' tokenizes the tweet into individual words, removes stop words and applies stemming.

In [8]:
# test the function below
print('This is an example of a positive tweet: \n', train_x[0])
print('\nThis is an example of the processed version of the tweet: \n', process_tweet(train_x[0]))

This is an example of a positive tweet: 
 :((((( matt

This is an example of the processed version of the tweet: 
 [':(', 'matt']


## 1 - Logistic Regression 
### 1.1 - Sigmoid
You will learn to use logistic regression for text classification.  
* The sigmoid function is defined as:  
  
$$ h(z) = \frac{1}{1 + \exp^{-z}} \tag{1}$$  

It maps the input 'z' to a value that ranges between 0 and 1, and so it can be treated as a probability.  
<div style="width:image width px; font-size:100%; text-align:center;"><img src='./sigmoid_plot.jpg' alt="alternate text" width="width" height="height" style="width:300px;height:200px;" /> Figure 1 </div>

### Exercise 1 -  sigmoid
Implement the sigmoid function. 
* You will want this function to work if z is a scalar as well as if it is an array.

In [9]:
# UNQ_C1 GRADED FUNCTION: sigmoid
def sigmoid(z): 
    '''
    Input:
        z: is the input (can be a scalar or an array)
    Output:
        h: the sigmoid of z
    '''
    h = 1/(1+np.exp(-z))
    return h

In [10]:
# Testing your function 
if (sigmoid(0) == 0.5):
    print('SUCCESS!')
else:
    print('Oops!')

if (sigmoid(4.92) == 0.9927537604041685):
    print('CORRECT!')
else:
    print('Oops again!')

SUCCESS!
CORRECT!


In [11]:
# Test your function
ut.test_sigmoid(sigmoid)

 All tests passed


### Exercise 2 - gradientDescent
Implement gradient descent function.

 $h(z) = sigmoid(z)$  
 $J = \frac{-1}{m} \times \left(\mathbf{y}^T \cdot log(\mathbf{h}) + \mathbf{(1-y)}^T \cdot log(\mathbf{1-h}) \right)$  
 $\mathbf{\theta} = \mathbf{\theta} - \frac{\alpha}{m} \times \left( \mathbf{x}^T \cdot \left( \mathbf{h-y} \right) \right)$

In [12]:
# UNQ_C2 GRADED FUNCTION: gradientDescent
def gradientDescent(x, y, theta, alpha, num_iters):
    '''
    Input:
        x: matrix of features which is (m,n+1)
        y: corresponding labels of the input matrix x, dimensions (m,1)
        theta: weight vector of dimension (n+1,1)
        alpha: learning rate
        num_iters: number of iterations you want to train your model for
    Output:
        J: the final cost
        theta: your final weight vector
    Hint: you might want to print the cost to make sure that it is going down.
    '''

    # get 'm', the number of rows in matrix x
    m = x.shape[0]
    
    for i in range(0, num_iters):
        
        # get z, the dot product of x and theta
        z = x.dot(theta)
        
        # get the sigmoid of z
        h = sigmoid(z)
        
        # calculate the cost function
        J = (-1/m)*(y.T.dot(np.log(h)) + (1-y).T.dot(np.log(1-h)))

        # update the weights theta
        theta = theta - alpha/m*(x.T.dot(h-y))
        
  
    J = float(J)
    return J, theta

In [13]:
# Check the function
# Construct a synthetic test case using numpy PRNG functions
np.random.seed(1)
# X input is 10 x 3 with ones for the bias terms
tmp_X = np.append(np.ones((10, 1)), np.random.rand(10, 2) * 2000, axis=1)
# Y Labels are 10 x 1
tmp_Y = (np.random.rand(10, 1) > 0.35).astype(float)

# Apply gradient descent
tmp_J, tmp_theta = gradientDescent(tmp_X, tmp_Y, np.zeros((3, 1)), 1e-8, 700)
print(f"The cost after training is {tmp_J:.8f}.")
print(f"The resulting vector of weights is {[round(t, 8) for t in np.squeeze(tmp_theta)]}")

The cost after training is 0.67094970.
The resulting vector of weights is [4.1e-07, 0.00035658, 7.309e-05]


C:\Users\mrinm\AppData\Local\Temp\ipykernel_17772\823394018.py:34: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  J = float(J)


In [14]:
# Test your function
ut.test_gradientDescent(gradientDescent)

 All tests passed


C:\Users\mrinm\AppData\Local\Temp\ipykernel_17772\823394018.py:34: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  J = float(J)


### 2 - Extracting the Features

* Given a list of tweets, extract the features and store them in a matrix. You will extract two features.
    * The first feature is the number of positive words in a tweet.
    * The second feature is the number of negative words in a tweet. 
* Then train your logistic regression classifier on these features.
* Test the classifier on a validation set. 

In [15]:
# UNQ_C3 GRADED FUNCTION: extract_features
def extract_features(tweet, freqs, process_tweet=process_tweet):
    '''
    Input: 
        tweet: a list of words for one tweet
        freqs: a dictionary corresponding to the frequencies of each tuple (word, label)
    Output: 
        x: a feature vector of dimension (1,3)
    '''
    # process_tweet tokenizes, stems, and removes stopwords
    word_l = process_tweet(tweet)
    
    # 3 elements for [bias, positive, negative] counts
    x = np.zeros(3) 
    
    # bias term is set to 1
    x[0] = 1 
    
    ### START CODE HERE ###
    
    # loop through each word in the list of words
    for word in word_l:
        
        # increment the word count for the positive label 1
        x[1] += freqs.get((word, 1), 0)
        
        # increment the word count for the negative label 0
        x[2] += freqs.get((word, -1), 0)
        
    ### END CODE HERE ###
    
    x = x[np.newaxis, :]  # adding batch dimension for further processing
    assert(x.shape == (1, 3))
    return x

In [16]:
# Check your function
# test 1
# test on training data
tmp1 = extract_features(train_x[0], freqs)
print(tmp1)

[[1. 6. 0.]]


In [17]:
# Test your function
ut.test_extract_features(extract_features, freqs)

Wrong output values. Check how you are computing the positive or negative word count. 
	Expected: [[1.000e+00 3.133e+03 6.100e+01]].
	Got: [[1.000e+00 3.097e+03 0.000e+00]].
Wrong output values. Check how you are computing the positive or negative word count. 
	Expected: [[  1. 263. 106.]].
	Got: [[  1. 258.   0.]].
Wrong output values. Check how you are computing the positive or negative word count. 
	Expected: [[  1.   5. 100.]].
	Got: [[1. 5. 0.]].
 5  Tests passed
 3  Tests failed


### 3 - Training Your Model

To train the model:
* Stack the features for all training examples into a matrix X. 
* Call `gradientDescent`, which you've implemented above.

In [18]:
# collect the features 'x' and stack them into a matrix 'X'
X = np.zeros((len(train_x), 3))
for i in range(len(train_x)):
    X[i, :]= extract_features(train_x[i], freqs)

# training labels corresponding to X
Y = train_y

# Apply gradient descent
J, theta = gradientDescent(X, Y, np.zeros((3, 1)), 1e-9, 1500)
print(f"The cost after training is {J:.8f}.")
print(f"The resulting vector of weights is {[round(t, 8) for t in np.squeeze(theta)]}")

The cost after training is 0.51558936.
The resulting vector of weights is [-1.4e-07, 0.00048455, 0.0]


C:\Users\mrinm\AppData\Local\Temp\ipykernel_17772\823394018.py:34: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  J = float(J)


### 4 -  Test your Logistic Regression

It is time for you to test your logistic regression function on some new input that your model has not seen before.

Implement `predict_tweet`.
Predict whether a tweet is positive or negative.

* Given a tweet, process it, then extract the features.
* Apply the model's learned weights on the features to get the logits.
* Apply the sigmoid to the logits to get the prediction (a value between 0 and 1).

$$y_{pred} = sigmoid(\mathbf{x} \cdot \theta)$$

In [19]:
def predict_tweet(tweet, freqs, theta):
    '''
    Input: 
        tweet: a string
        freqs: a dictionary corresponding to the frequencies of each tuple (word, label)
        theta: (3,1) vector of weights
    Output: 
        y_pred: the probability of a tweet being positive or negative
    '''
    ### START CODE HERE ###
    
    # extract the features of the tweet and store it into x
    x = extract_features(tweet, freqs, process_tweet=process_tweet)
    
    # make the prediction using x and theta
    y_pred = sigmoid(x.dot(theta))
    
    ### END CODE HERE ###
    
    return y_pred

In [20]:
# Run this cell to test your function
for tweet in ['I am happy', 'I am bad', 'this movie should have been great.', 'great', 'great great', 'great great great', 'great great great great']:
    print( '%s -> %f' % (tweet, predict_tweet(tweet, freqs, theta)))    
    

I am happy -> 0.521065
I am bad -> 0.501696
this movie should have been great. -> 0.519977
great -> 0.517558
great great -> 0.535072
great great great -> 0.552500
great great great great -> 0.569800


C:\Users\mrinm\AppData\Local\Temp\ipykernel_17772\1003351.py:3: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  print( '%s -> %f' % (tweet, predict_tweet(tweet, freqs, theta)))


In [ ]:
ut.test_predict_tweet(predict_tweet, freqs, theta)


#### 4.1 -  Check the Performance using the Test Set
After training your model using the training set above, check how your model might perform on real, unseen data, by testing it against the test set.

Implement `test_logistic_regression`. 
* Given the test data and the weights of your trained model, calculate the accuracy of your logistic regression model. 
* Use your 'predict_tweet' function to make predictions on each tweet in the test set.
* If the prediction is > 0.5, set the model's classification 'y_hat' to 1, otherwise set the model's classification 'y_hat' to 0.
* A prediction is accurate when the y_hat equals the test_y.  Sum up all the instances when they are equal and divide by m.


In [27]:
# UNQ_C5 GRADED FUNCTION: test_logistic_regression
def test_logistic_regression(test_x, test_y, freqs, theta, predict_tweet=predict_tweet):
    """
    Input: 
        test_x: a list of tweets
        test_y: (m, 1) vector with the corresponding labels for the list of tweets
        freqs: a dictionary with the frequency of each pair (or tuple)
        theta: weight vector of dimension (3, 1)
    Output: 
        accuracy: (# of tweets classified correctly) / (total # of tweets)
    """
    
    ### START CODE HERE ###
    
    # the list for storing predictions
    y_hat = []
    
    for tweet in test_x:
        # get the label prediction for the tweet
        y_pred = predict_tweet(tweet, freqs, theta)
        
        if y_pred > 0.5:
            # append 1.0 to the list
            y_hat.append(1.0)
        else:
            # append 0 to the list
            y_hat.append(0.0)

    # With the above implementation, y_hat is a list, but test_y is (m,1) array
    # convert both to one-dimensional arrays in order to compare them using the '==' operator
    
    accuracy = (np.sum(np.array(y_hat).reshape(len(y_hat), 1) == test_y))/test_y.size

    ### END CODE HERE ###
    
    return accuracy

In [28]:
tmp_accuracy = test_logistic_regression(test_x, test_y, freqs, theta)
print(f"Logistic regression model's accuracy = {tmp_accuracy:.4f}")

Logistic regression model's accuracy = 0.5070


### 5 -  Error Analysis

In this part you will see some tweets that your model misclassified. Why do you think the misclassifications happened? Specifically what kind of tweets does your model misclassify?

In [29]:
# Some error analysis done for you
print('Label Predicted Tweet')
for x,y in zip(test_x,test_y):
    y_hat = predict_tweet(x, freqs, theta)

    if np.abs(y - (y_hat > 0.5)) > 0:
        print('THE TWEET IS:', x)
        print('THE PROCESSED TWEET IS:', process_tweet(x))
        print('%d\t%0.8f\t%s' % (y, y_hat, ' '.join(process_tweet(x)).encode('ascii', 'ignore')))

Label Predicted Tweet
THE TWEET IS: I love you, how but you? @Taecyeon2pm8 did you feel the same? Emm I think not :(
THE PROCESSED TWEET IS: ['love', 'feel', 'emm', 'think', ':(']
0	0.54962310	b'love feel emm think :('
THE TWEET IS: @scottybev I'm not surprised, that sounds hellish! Why would you do such a thing? :(
THE PROCESSED TWEET IS: ["i'm", 'surpris', 'sound', 'hellish', 'would', 'thing', ':(']
0	0.53579498	b"i'm surpris sound hellish would thing :("
THE TWEET IS: @hanbined sad pray for me :(((
THE PROCESSED TWEET IS: ['sad', 'pray', ':(']
0	0.50072678	b'sad pray :('
THE TWEET IS: Popol day too :(
THE PROCESSED TWEET IS: ['popol', 'day', ':(']
0	0.52493335	b'popol day :('
THE TWEET IS: @caylahhhh lmfao seriously??? I can't remember if I did honestly..more likely tho :((
THE PROCESSED TWEET IS: ['lmfao', 'serious', "can't", 'rememb', 'honestli', '..', 'like', 'tho', ':(']
0	0.54145435	b"lmfao serious can't rememb honestli .. like tho :("
THE TWEET IS: Should have taken a pic befo

C:\Users\mrinm\AppData\Local\Temp\ipykernel_17772\289099189.py:9: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  print('%d\t%0.8f\t%s' % (y, y_hat, ' '.join(process_tweet(x)).encode('ascii', 'ignore')))


0	0.50472414	b"dream went colleg fail anim :( :( :( :'("
THE TWEET IS: @WforWoman 
9) shopping will be like fries without ketchup. Tasteless :(

#WSaleLove
THE PROCESSED TWEET IS: ['9', 'shop', 'like', 'fri', 'without', 'ketchup', 'tasteless', ':(', 'wsalelov']
0	0.52698711	b'9 shop like fri without ketchup tasteless :( wsalelov'
THE TWEET IS: youngjae is getting more handsome im :(((
THE PROCESSED TWEET IS: ['youngja', 'get', 'handsom', 'im', ':(']
0	0.52553749	b'youngja get handsom im :('
THE TWEET IS: i slept all day and now i can't sleep :(
THE PROCESSED TWEET IS: ['slept', 'day', "can't", 'sleep', ':(']
0	0.53302204	b"slept day can't sleep :("
THE TWEET IS: @Uber How do we get our free Cornettos? We can't figure it out :-( x
THE PROCESSED TWEET IS: ['get', 'free', 'cornetto', "can't", 'figur', ':-(', 'x']
0	0.53458962	b"get free cornetto can't figur :-( x"
THE TWEET IS: 30 minutes and counting just to pass through the EDSA AYALA tunnel... And am still not completely out. :(
THE PR

### 6 - Predict with your own Tweet

In [30]:
# Feel free to change the tweet below
my_tweet = 'This is a ridiculously bright movie. The plot was terrible and I was sad until the ending!'
print(process_tweet(my_tweet))
y_hat = predict_tweet(my_tweet, freqs, theta)
print(y_hat)
if y_hat > 0.5:
    print('Positive sentiment')
else: 
    print('Negative sentiment')

['ridicul', 'bright', 'movi', 'plot', 'terribl', 'sad', 'end']
[[0.50545088]]
Positive sentiment
